# Browser-Use Download Functionality Tutorial

## Overview

This tutorial demonstrates how to enable file download capabilities in Browser-Use for remote browser environments. The download patch allows agents to download files using HTTP requests instead of relying on browser file dialogs, which don't work in headless/remote scenarios.

### Tutorial Details

| Information         | Details                                                                                   |
|:--------------------|:-------------------------------------------------------------------------------------------|
| Tutorial type       | Enhancement                                                                               |
| Agent type          | Single                                                                                    |
| Agentic Framework   | Browser-Use Enhanced                                                                      |
| LLM model           | Anthropic Claude 3.7 Sonnet                                                              |
| Tutorial components | Download patch installation, remote browser file downloads, agent download awareness      |
| Tutorial vertical   | Cross-vertical                                                                            |
| Example complexity  | Intermediate                                                                              |
| SDK used            | Amazon BedrockAgentCore Python SDK, Browser-Use Enhanced                                 |

### Key Features

* **Remote Browser Downloads**: Download files in headless/remote browser environments
* **Agent Download Awareness**: LLM sees download progress and failures in real-time
* **HTTP Client Fallback**: Uses HTTP requests when browser dialogs fail
* **Progress Tracking**: Monitor download progress and handle failures gracefully
* **Backward Compatible**: Works with existing Browser-Use code

## Prerequisites

To execute this tutorial you will need:
* Python 3.11+
* browser-use==0.11.2 (specific version required for patch compatibility)
* bedrock-agentcore
* Valid AWS credentials configured

## 1. Installation and Setup

### 1.1 Install Dependencies

In [ ]:
# Install specific browser-use version for patch compatibility
!pip install browser-use==0.11.2 bedrock-agentcore boto3 --quiet

### 1.2 Apply Download Functionality Patch

The download patch enhances browser-use with remote download capabilities.

In [ ]:
%%writefile apply_download_patch.py
#!/usr/bin/env python3
"""
Browser-Use Download Functionality Patch
Applies download enhancements to browser-use 0.11.2 installation
"""

import os
import sys
import tempfile
import zipfile
import shutil
import subprocess
from pathlib import Path

def find_browser_use_installation():
    """Find the browser-use installation path"""
    try:
        result = subprocess.run([sys.executable, "-c", "import browser_use; print(browser_use.__file__)"], 
                              capture_output=True, text=True, check=True)
        browser_use_init = result.stdout.strip()
        return Path(browser_use_init).parent
    except subprocess.CalledProcessError:
        print("❌ Error: browser-use not found. Please install browser-use==0.11.2 first")
        sys.exit(1)

def apply_patch():
    """Apply the download functionality patch"""
    print("🔧 Applying browser-use download functionality patch...")
    
    # Find installation path
    install_path = find_browser_use_installation()
    print(f"📍 Found browser-use installation: {install_path}")
    
    # Get script directory and zip file
    script_dir = Path(__file__).parent
    zip_file = script_dir / "download-patch.zip"
    
    if not zip_file.exists():
        print(f"❌ Error: {zip_file} not found")
        sys.exit(1)
    
    # Create temp directory and extract
    with tempfile.TemporaryDirectory() as temp_dir:
        print("📦 Extracting patch files...")
        with zipfile.ZipFile(zip_file, 'r') as zip_ref:
            zip_ref.extractall(temp_dir)
        
        patch_dir = Path(temp_dir) / "download-patch"
        
        # File mapping: source -> destination
        file_mapping = {
            "downloads_watchdog.py": "browser/watchdogs/downloads_watchdog.py",
            "download_manager.py": "browser/download_manager.py", 
            "session.py": "browser/session.py",
            "profile.py": "browser/profile.py",
            "message_manager_service.py": "agent/message_manager/service.py",
            "prompts.py": "agent/prompts.py",
            "agent_service.py": "agent/service.py"
        }
        
        # Copy files
        for source_file, dest_path in file_mapping.items():
            source = patch_dir / source_file
            destination = install_path / dest_path
            
            if not source.exists():
                print(f"❌ Warning: {source_file} not found in patch")
                continue
                
            print(f"📄 Copying {source_file}")
            print(f"   From: {source}")
            print(f"   To:   {destination}")
            shutil.copy2(source, destination)
    
    print("✅ Download functionality patch applied successfully!")
    print("\n🎯 Usage:")
    print("  Set download_from_remote_browser=True in your browser profile")
    print("  Downloads will use HTTP client instead of browser file dialogs")

if __name__ == "__main__":
    apply_patch()

In [ ]:
# Apply the download functionality patch
!python apply_download_patch.py

## 2. Basic Download Usage

### 2.1 Simple Download Example

In [ ]:
import asyncio
from browser_use import Agent, BrowserProfile
from bedrock_agentcore.client import BedrockAgentCoreClient

async def basic_download_example():
    """Basic example of downloading a file with enhanced browser-use"""
    
    # Create browser profile with download enhancement enabled
    profile = BrowserProfile(
        download_from_remote_browser=True,  # Enable HTTP download fallback
        headless=True,
        downloads_path="./downloads"
    )
    
    # Initialize Bedrock client
    client = BedrockAgentCoreClient()
    
    # Create agent with download-enhanced profile
    agent = Agent(
        task="Go to https://www.w3.org/WAI/ER/tests/xhtml/testfiles/resources/pdf/dummy.pdf and download the PDF file",
        llm=client.get_bedrock_llm(model_id="anthropic.claude-3-5-sonnet-20241022-v2:0"),
        browser_profile=profile
    )
    
    # Run the download task
    result = await agent.run()
    print(f"Download task completed: {result}")
    
    return result

# Run the example
await basic_download_example()

### 2.2 Download with Progress Monitoring

In [ ]:
async def download_with_monitoring():
    """Example showing download progress monitoring and agent awareness"""
    
    profile = BrowserProfile(
        download_from_remote_browser=True,
        headless=True,
        downloads_path="./downloads"
    )
    
    client = BedrockAgentCoreClient()
    
    agent = Agent(
        task="""Go to a website with downloadable files and download a PDF. 
        Monitor the download progress and let me know when it's complete.
        If the download fails, try an alternative approach.""",
        llm=client.get_bedrock_llm(model_id="anthropic.claude-3-5-sonnet-20241022-v2:0"),
        browser_profile=profile
    )
    
    # The agent will now see download progress in its context:
    # <downloads_in_progress>Downloading: file.pdf (45%, 12s elapsed)</downloads_in_progress>
    # <failed_downloads>Failed: file.pdf (Network error) 3m ago</failed_downloads>
    
    result = await agent.run()
    print(f"Monitored download completed: {result}")
    
    return result

# Run the monitoring example
await download_with_monitoring()

## 3. Troubleshooting and Best Practices

### 3.1 Verify Installation

In [ ]:
# Check if download patch is properly applied
def verify_download_patch():
    """Verify that the download functionality is properly installed"""
    try:
        from browser_use.browser.download_manager import DownloadManager
        from browser_use import BrowserProfile
        
        # Check if download_from_remote_browser parameter exists
        profile = BrowserProfile(download_from_remote_browser=True)
        
        print("✅ Download functionality patch is properly installed")
        print(f"✅ download_from_remote_browser setting: {profile.download_from_remote_browser}")
        return True
        
    except ImportError as e:
        print(f"❌ Download patch not properly installed: {e}")
        return False
    except Exception as e:
        print(f"❌ Error verifying patch: {e}")
        return False

# Verify installation
verify_download_patch()

### 3.2 Configuration Options

The download enhancement adds the following configuration option to `BrowserProfile`:

- **`download_from_remote_browser`** (bool, default: False)
  - `True`: Use HTTP client for downloads (recommended for remote/headless browsers)
  - `False`: Use standard browser download behavior (works for local browsers with file dialogs)

### 3.3 Best Practices

1. **Always set `download_from_remote_browser=True`** for headless or remote browser scenarios
2. **Specify a `downloads_path`** to control where files are saved
3. **Monitor agent context** - the LLM will see download progress and can make intelligent decisions
4. **Handle failures gracefully** - the agent can retry with different approaches if downloads fail

## 4. Real-World Example: Document Processing Pipeline

In [ ]:
import os
from pathlib import Path

async def document_processing_pipeline():
    """Real-world example: Download and process multiple documents"""
    
    # Ensure downloads directory exists
    downloads_dir = Path("./downloads")
    downloads_dir.mkdir(exist_ok=True)
    
    profile = BrowserProfile(
        download_from_remote_browser=True,
        headless=True,
        downloads_path=str(downloads_dir)
    )
    
    client = BedrockAgentCoreClient()
    
    agent = Agent(
        task="""I need you to:
        1. Go to a website that has PDF documents available for download
        2. Download 2-3 different PDF files
        3. For each download, monitor the progress and confirm successful completion
        4. If any download fails, try an alternative approach or find a different file
        5. List all successfully downloaded files at the end
        
        Make sure to use the HTTP download method since we're in a remote browser environment.""",
        llm=client.get_bedrock_llm(model_id="anthropic.claude-3-5-sonnet-20241022-v2:0"),
        browser_profile=profile
    )
    
    result = await agent.run()
    
    # List downloaded files
    downloaded_files = list(downloads_dir.glob("*.pdf"))
    print(f"\n📁 Downloaded {len(downloaded_files)} files:")
    for file in downloaded_files:
        size_mb = file.stat().st_size / (1024 * 1024)
        print(f"  - {file.name} ({size_mb:.2f} MB)")
    
    return result, downloaded_files

# Run the document processing pipeline
result, files = await document_processing_pipeline()
print(f"\nPipeline completed with {len(files)} files downloaded")

## Conclusion

This tutorial demonstrated how to enhance Browser-Use with download functionality for remote browser environments. Key takeaways:

- **Download patch enables file downloads** in headless/remote browser scenarios
- **Agent awareness** allows the LLM to monitor download progress and handle failures
- **HTTP client fallback** works when browser file dialogs are not available
- **Backward compatible** with existing Browser-Use applications

The enhanced download functionality is particularly useful for:
- Document processing workflows
- Data collection tasks
- Remote browser automation
- Headless browser scenarios

For more advanced usage and customization options, refer to the Browser-Use documentation and the download patch source code.